
## 1. Event Data Exploration

### Objective
Explore the structure of StatsBomb event-level data before building player-level performance metrics.

### Data Structure
Each row in the `events` DataFrame represents an event recorded during a football match.

The main attributes include:
- `id`: Unique event identifier.
- `minute`, `second`: Time of the event.
- `team`, `player`: Team and player associated with the event.
- `type`: Type of event (pass, shot, pressure, etc.).
- `location`: Starting coordinates of the event on the pitch.

### Initial Observations
- A single match contains thousands of recorded events.
- Events include passes, shots, pressures, ball recoveries, and other actions.
- Some events, such as `Starting XI`, do not have an associated player or pitch location.
- Missing values (`NaN`) depend on the event type and do not necessarily indicate data quality issues.

These observations will guide the data cleaning and feature engineering stages.


In [14]:

from statsbombpy import sb

matches = sb.matches(
    competition_id=2,
    season_id=27
)

print(matches.shape)
matches.head()


(380, 55)


/Users/bilal/Desktop/football-player-similarity/.venv/lib/python3.14/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,match_id,match_date,kick_off,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,match_week,...,home_manager_country_name,away_manager_id,away_manager_name,away_manager_nickname,away_manager_dob,away_manager_country_id,away_manager_country_name,data_version,shot_fidelity_version,xy_fidelity_version
0,3754217,2015-09-19,11:45:00.000,2,0,available,scheduled,2025-12-16T17:01:18.696515,2021-06-13T16:17:31.694,6,...,Portugal,577,Arsène Wenger,NaN,1949-10-22,78,France,1.1.0,2,2
1,3754117,2015-12-13,12:30:00.000,0,2,available,scheduled,2025-12-16T17:04:24.176407,2021-06-13T16:17:31.694,16,...,France,577,Arsène Wenger,NaN,1949-10-22,78,France,1.1.0,2,2
2,3754296,2015-12-21,19:00:00.000,2,1,available,scheduled,2025-12-17T11:59:35.059270,2021-06-13T16:17:31.694,17,...,France,733,Manuel Luis Pellegrini Ripamonti,Manuel Pellegrini,1953-09-16,45,Chile,1.1.0,2,2
3,3753983,2015-10-31,14:00:00.000,0,3,available,scheduled,2025-12-16T17:02:24.849544,2021-06-13T16:17:31.694,11,...,England,577,Arsène Wenger,NaN,1949-10-22,78,France,1.1.0,2,2
4,3754160,2015-12-05,14:00:00.000,3,1,available,scheduled,2025-12-16T17:03:25.661537,2021-06-13T16:17:31.694,15,...,France,561,Sam Allardyce,NaN,1954-10-19,68,England,1.1.0,2,2


In [15]:

match_id = int(matches.iloc[0]["match_id"])

events = sb.events(match_id=match_id)

print(events.shape)
events["type"].value_counts().head(10)


/Users/bilal/Desktop/football-player-similarity/.venv/lib/python3.14/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


(3732, 90)


type
Pass             1046
Ball Receipt*     963
Carry             814
Pressure          320
Ball Recovery     106
Duel               78
Dribble            52
Clearance          42
Block              41
Goal Keeper        34
Name: count, dtype: int64

In [16]:

events[[
    "id",
    "minute",
    "second",
    "team",
    "player",
    "type",
    "location"
]].head(10)


,id,minute,second,team,player,type,location
0,9d86a178-3514-45d1-9d14-1372e846d17b,0,0,Chelsea,NaN,Starting XI,NaN
1,82acc213-90f3-4fee-8305-bd9e403cec42,0,0,Arsenal,NaN,Starting XI,NaN
2,d18f1a33-65ba-42e5-a9fd-bbfc709694e8,0,0,Chelsea,NaN,Half Start,NaN
3,49147091-455c-4372-8d56-96e1ca71d01a,0,0,Arsenal,NaN,Half Start,NaN
4,7457b6d0-10f8-48ea-9b80-8c0703914ece,45,0,Arsenal,NaN,Half Start,NaN
5,d9259d49-479e-4faa-aa7c-5b8fc586c8b2,45,0,Chelsea,NaN,Half Start,NaN
6,5f9bdb6d-0379-4f42-9f53-29777f166db5,0,0,Arsenal,Theo Walcott,Pass,"[61.0, 40.1]"
7,e5d7db8c-d134-4392-a109-9f6efc7f7225,0,1,Arsenal,Mesut Özil,Pass,"[59.3, 42.1]"
8,fd5039d0-8dd4-417c-9eac-decd2d267e93,0,5,Arsenal,Santiago Cazorla González,Pass,"[49.5, 41.7]"
9,811b65c7-c0b1-442c-a412-bff5fcf98ff5,0,7,Arsenal,Aaron Ramsey,Pass,"[65.9, 48.7]"



## 2. Pass Event Analysis

### Objective
Examine passing events to understand how StatsBomb records ball distribution and passing outcomes.

### Key Variables
- `player`: Player attempting the pass.
- `team`: Team performing the action.
- `location`: Starting coordinates of the pass.
- `pass_end_location`: Ending coordinates of the pass.
- `pass_outcome`: Outcome of the pass.

### Methodology
We filter the event dataset to retain only `Pass` events using Pandas boolean indexing.

A missing value (`NaN`) in `pass_outcome` generally indicates a successful pass, while other values describe unsuccessful or alternative outcomes.

### Future Applications
These variables will be used to compute:
- Passes attempted per 90 minutes.
- Pass completion percentage.
- Progressive passes per 90 minutes.
- Passing distance and ball progression metrics.


In [17]:
passes = events.loc[events["type"] == "Pass"].copy()

passes[["minute", "player", "team", "location", "pass_end_location", "pass_outcome"]].head(10)

,minute,player,team,location,pass_end_location,pass_outcome
6,0,Theo Walcott,Arsenal,"[61.0, 40.1]","[59.1, 42.1]",NaN
7,0,Mesut Özil,Arsenal,"[59.3, 42.1]","[43.7, 38.2]",NaN
8,0,Santiago Cazorla González,Arsenal,"[49.5, 41.7]","[63.1, 48.9]",NaN
9,0,Aaron Ramsey,Arsenal,"[65.9, 48.7]","[69.6, 48.9]",Incomplete
10,0,Francesc Fàbregas i Soler,Chelsea,"[47.1, 38.2]","[32.6, 39.3]",NaN
11,0,Kurt Happy Zouma,Chelsea,"[34.3, 44.2]","[51.0, 51.7]",NaN
12,0,Aaron Ramsey,Arsenal,"[78.8, 32.0]","[76.0, 33.1]",Incomplete
13,0,Mesut Özil,Arsenal,"[76.4, 39.5]","[83.5, 45.1]",Incomplete
14,0,Nemanja Matić,Chelsea,"[33.4, 36.5]","[6.9, 42.9]",NaN
15,0,Asmir Begović,Chelsea,"[9.3, 46.8]","[69.8, 63.9]",Incomplete


In [18]:
passes["pass_outcome"].value_counts(dropna=False).head(10)

pass_outcome
NaN                 863
Incomplete          155
Out                  15
Unknown               6
Injury Clearance      4
Pass Offside          3
Name: count, dtype: int64


## 3. Spatial Analysis of Passes

### Objective
Understand StatsBomb's pitch coordinate system and examine how passing events describe ball movement.

### Coordinate System
StatsBomb uses a 120 × 80 coordinate system:
- `x`: Position along the length of the pitch.
- `y`: Position across the width of the pitch.
- `location`: Starting position of the pass.
- `pass_end_location`: Ending position of the pass.

### Methodology
Extract the starting and ending coordinates of each pass and calculate the change in horizontal position.

A positive change in x indicates movement toward the opposition goal, while a negative change indicates backward movement.

This initial measure will help us explore ball progression before developing a more rigorous progressive-pass metric.


In [19]:
passes_spatial = passes.dropna(subset = ["location", "pass_end_location"]).copy()


passes_spatial["start_x"] = passes_spatial["location"].str[0]
passes_spatial["start_y"] = passes_spatial["location"].str[1]

passes_spatial["end_x"] = passes_spatial["pass_end_location"].str[0]
passes_spatial["end_y"] = passes_spatial["pass_end_location"].str[1]

passes_spatial["delta_x"] = (
    passes_spatial["end_x"] - passes_spatial["start_x"]
)

passes_spatial[[
    "player", "start_x", "start_y",
    "end_x", "end_y", "delta_x"
]].head(10)

,player,start_x,start_y,end_x,end_y,delta_x
6,Theo Walcott,61.0,40.1,59.1,42.1,-1.9
7,Mesut Özil,59.3,42.1,43.7,38.2,-15.6
8,Santiago Cazorla González,49.5,41.7,63.1,48.9,13.6
9,Aaron Ramsey,65.9,48.7,69.6,48.9,3.7
10,Francesc Fàbregas i Soler,47.1,38.2,32.6,39.3,-14.5
11,Kurt Happy Zouma,34.3,44.2,51.0,51.7,16.7
12,Aaron Ramsey,78.8,32.0,76.0,33.1,-2.8
13,Mesut Özil,76.4,39.5,83.5,45.1,7.1
14,Nemanja Matić,33.4,36.5,6.9,42.9,-26.5
15,Asmir Begović,9.3,46.8,69.8,63.9,60.5



## 4. Player-Level Passing Statistics

### Objective
Aggregate individual passing events to calculate player-level performance metrics.

### Methodology
We select all passes attempted by a specific player and calculate:
- Total passes attempted.
- Total completed passes.
- Pass completion percentage.

A pass is considered completed when `pass_outcome` is missing (`NaN`), following StatsBomb's event data convention.

### Limitations
These statistics describe a single match, not an entire season. Season-level metrics will require aggregating events across multiple matches and accounting for playing time.


In [20]:

player_name = "Mesut Özil"

player_passes = passes.loc[
    passes["player"] == player_name
].copy()

player_passes["is_completed"] = (
    player_passes["pass_outcome"].isna()
)

total_passes = len(player_passes)
completed_passes = player_passes["is_completed"].sum()

completion_rate = (
    completed_passes / total_passes * 100
)

print(f"Player: {player_name}")
print(f"Passes attempted: {total_passes}")
print(f"Passes completed: {completed_passes}")
print(f"Completion rate: {completion_rate:.2f}%")


Player: Mesut Özil
Passes attempted: 33
Passes completed: 28
Completion rate: 84.85%



## 5. Aggregating Player-Level Passing Statistics

### Objective
Calculate passing statistics for every player in a match using Pandas groupby operations.

### Methodology
We group passing events by team and player, then aggregate the data to calculate:
- Total passes attempted.
- Total completed passes.
- Pass completion percentage.

### Expected Output
A player-level dataset containing one row per player, with aggregated passing statistics.

This approach will later be extended to calculate season-level performance metrics.


In [21]:

passes_all = passes.copy()

passes_all["is_completed"] = passes_all["pass_outcome"].isna()

player_stats = (
    passes_all.groupby(["team", "player"], as_index=False)
    .agg(
        passes_attempted=("id", "size"),
        passes_completed=("is_completed", "sum")
    )
)

player_stats["completion_rate"] = (
    player_stats["passes_completed"]
    / player_stats["passes_attempted"] * 100
)

player_stats.sort_values(
    "passes_attempted", ascending=False
).head(10)


,team,player,passes_attempted,passes_completed,completion_rate
19,Chelsea,Francesc Fàbregas i Soler,90,73,81.111111
16,Chelsea,César Azpilicueta Tanco,85,73,85.882353
24,Chelsea,Nemanja Matić,83,77,92.771084
18,Chelsea,Eden Hazard,74,69,93.243243
12,Arsenal,Santiago Cazorla González,62,50,80.645161
26,Chelsea,Pedro Eliezer Rodríguez Ledesma,60,54,90.000000
0,Arsenal,Aaron Ramsey,47,31,65.957447
15,Chelsea,Branislav Ivanović,46,37,80.434783
6,Arsenal,Héctor Bellerín Moruno,43,28,65.116279
7,Arsenal,Ignacio Monreal Eraso,43,36,83.720930



## 6. Shot Event Exploration

### Objective
Explore shot events and identify the variables used to measure offensive performance.

### Key Variables
- `player`: Player attempting the shot.
- `team`: Team associated with the shot.
- `location`: Shot location on the pitch.
- `shot_outcome`: Outcome of the shot.
- `shot_statsbomb_xg`: Expected goals value assigned to the shot.

### Future Applications
These variables will help calculate shooting frequency, shot quality, and expected goals at the player level.


In [22]:

shots = events.loc[
    events["type"] == "Shot"
].copy()

shots[[
    "player",
    "team",
    "minute",
    "location",
    "shot_outcome",
    "shot_statsbomb_xg"
]].head(10)


,player,team,minute,location,shot_outcome,shot_statsbomb_xg
3547,Aaron Ramsey,Arsenal,1,"[94.5, 42.9]",Blocked,0.038832
3548,Francesc Fàbregas i Soler,Chelsea,5,"[93.5, 48.4]",Saved,0.031541
3549,Alexis Alejandro Sánchez Sánchez,Arsenal,11,"[89.8, 22.3]",Off T,0.006660
3550,Diego da Silva Costa,Chelsea,23,"[98.2, 26.0]",Saved,0.022902
3551,Theo Walcott,Arsenal,28,"[105.7, 24.2]",Saved,0.049741
3552,Pedro Eliezer Rodríguez Ledesma,Chelsea,31,"[115.1, 45.5]",Saved,0.400975
3553,Aaron Ramsey,Arsenal,34,"[103.9, 23.5]",Off T,0.023207
3554,Kurt Happy Zouma,Chelsea,36,"[87.3, 42.5]",Off T,0.020538
3555,Pedro Eliezer Rodríguez Ledesma,Chelsea,37,"[92.6, 47.4]",Off T,0.033888
3556,Francesc Fàbregas i Soler,Chelsea,38,"[95.7, 28.2]",Blocked,0.026702



## 7. Player-Level Shooting Statistics

### Objective
Aggregate shot events to calculate individual attacking performance metrics.

### Methodology
Using Pandas groupby and aggregation operations, we calculate the following statistics for each player:

- `shots`: Total number of shots attempted.
- `goals`: Total number of goals scored.
- `total_xg`: Sum of expected goals across all shots.
- `avg_xg_per_shot`: Average expected goals per shot.

### Interpretation
Expected goals (xG) measure the estimated quality of scoring opportunities.

Total xG represents the sum of the estimated scoring probabilities across a player's shots, while average xG per shot measures the average quality of those opportunities.

### Limitations
These metrics describe a single match. Season-level comparisons will require aggregation across matches and normalization by playing time.


In [23]:

# Identify goals
shots["is_goal"] = shots["shot_outcome"] == "Goal"

# Aggregate statistics by player
player_shots = (
    shots.groupby(["team", "player"], as_index=False)
    .agg(
        shots=("id", "size"),
        goals=("is_goal", "sum"),
        total_xg=("shot_statsbomb_xg", "sum"),
        avg_xg_per_shot=("shot_statsbomb_xg", "mean")
    )
)

# Display players with the highest total xG
player_shots.sort_values(
    "total_xg", ascending=False
).head(10)


,team,player,shots,goals,total_xg,avg_xg_per_shot
10,Chelsea,Pedro Eliezer Rodríguez Ledesma,5,0,0.603349,0.120670
2,Arsenal,Theo Walcott,4,0,0.425706,0.106426
5,Chelsea,Eden Hazard,4,0,0.302334,0.075584
0,Arsenal,Aaron Ramsey,3,0,0.229568,0.076523
7,Chelsea,Kurt Happy Zouma,2,1,0.217984,0.108992
1,Arsenal,Alexis Alejandro Sánchez Sánchez,3,0,0.182253,0.060751
6,Chelsea,Francesc Fàbregas i Soler,4,0,0.168582,0.042145
4,Chelsea,Diego da Silva Costa,2,0,0.110855,0.055427
8,Chelsea,Loïc Rémy,1,0,0.093800,0.093800
11,Chelsea,Ramires Santos do Nascimento,1,0,0.076219,0.076219
